In [0]:
# MAGIC %md
# MAGIC ### Step 1: Initialize Workflow Parameters & Catalog
# MAGIC Setting up dynamic parameters for the Databricks Workflow.

dbutils.widgets.text("catalog_name", "smart-factory-catalog", "Unity Catalog Name")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze Schema Name")
dbutils.widgets.text("silver_schema", "silver", "Silver Schema Name")
dbutils.widgets.text("base_s3_path", "s3://smart-factory-analytics-bucket", "Base S3 Path")

catalog_name = dbutils.widgets.get("catalog_name")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
base_s3_path = dbutils.widgets.get("base_s3_path")

checkpoint_base = f"{base_s3_path}/unity-catalog/checkpoints/{silver_schema}/"

# Ensure Silver schema exists
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{silver_schema}`")

In [0]:
# MAGIC %md
# MAGIC ### Step 2: Define Target Silver Tables
# MAGIC We must explicitly create the Silver tables with Change Data Feed (CDF) enabled so the Gold layer can read them incrementally later.

def create_silver_table(stream_name: str):
    table_name = f"`{catalog_name}`.`{silver_schema}`.{stream_name}_telemetry_clean"
    location = f"{base_s3_path}/unity-catalog/tables/{silver_schema}/{stream_name}/"
    
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        event_id STRING,
        device_id STRING,
        device_type STRING,
        zone STRING,
        event_time TIMESTAMP,
        motor_current_amps DOUBLE,
        temperature_c DOUBLE,
        vibration_x DOUBLE,
        est_power_watts DOUBLE,
        is_temp_outlier BOOLEAN,
        is_vibration_outlier BOOLEAN,
        machine_health_status STRING,
        bronze_ingestion_timestamp TIMESTAMP,
        silver_processing_timestamp TIMESTAMP
    )
    USING DELTA
    LOCATION '{location}'
    TBLPROPERTIES ('delta.enableChangeDataFeed' = true)
    """)
    return table_name

In [0]:
spark.sql(f"""
-- Create the Quarantine table to hold failed records
CREATE TABLE IF NOT EXISTS `{catalog_name}`.`{silver_schema}`.quarantine_telemetry (
    event_id STRING,
    device_id STRING,
    device_type STRING,
    event_time TIMESTAMP,
    motor_current_amps DOUBLE,
    temperature_c DOUBLE,
    vibration_x DOUBLE,
    dq_failure_reason STRING,
    bronze_ingestion_timestamp TIMESTAMP
)
USING DELTA
LOCATION '{base_s3_path}/unity-catalog/tables/{silver_schema}/quarantine_telemetry/'
""")

In [0]:
# MAGIC %md
# MAGIC ### Step 3: Define the Transformation & Idempotent Merge Logic
# MAGIC This function executes on every micro-batch of new data arriving from Bronze.

from pyspark.sql.functions import col, to_timestamp, current_timestamp, when, lit
from delta.tables import DeltaTable

def process_silver_microbatch(microBatchDF, batchId, target_table_name):
    # 1. INITIAL CLEANUP
    df_clean = microBatchDF.withColumn("event_time", to_timestamp(col("event_time")))
    
    # 2. APPLY DATA QUALITY RULES
    # We evaluate physical limitations of the machinery and logical constraints
    df_evaluated = df_clean.withColumn(
        "dq_failure_reason",
        when(col("event_id").isNull() | col("device_id").isNull(), lit("Missing critical routing keys"))
        .when(col("event_time") > current_timestamp(), lit("Event time is in the future"))
        .when(col("temperature_c") < -50.0, lit("Temperature below physical limits (-50C)"))
        .when(col("temperature_c") > 1000.0, lit("Temperature exceeds physical limits (1000C)"))
        .when(col("motor_current_amps") < 0.0, lit("Motor current cannot be negative"))
        .when(col("vibration_x") < 0.0, lit("Vibration magnitude cannot be negative"))
        .otherwise(lit("PASS"))
    )

    # 3. BIFURCATE THE STREAM
    # Split the data into Good (PASS) and Bad (Quarantine)
    df_valid = df_evaluated.filter(col("dq_failure_reason") == "PASS").drop("dq_failure_reason")
    df_quarantine = df_evaluated.filter(col("dq_failure_reason") != "PASS")

    # 4. ROUTE BAD DATA TO QUARANTINE (Append Only)
    if df_quarantine.count() > 0:
        (df_quarantine
            .select("event_id", "device_id", "device_type", "event_time", 
                    "motor_current_amps", "temperature_c", "vibration_x", 
                    "dq_failure_reason", col("ingestion_timestamp").alias("bronze_ingestion_timestamp"))
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(f"`{catalog_name}`.`{silver_schema}`.quarantine_telemetry")
        )

    # 5. PROCESS VALID DATA (Deduplicate & Enrich)
    if df_valid.count() > 0:
        df_enriched = (df_valid
            .dropDuplicates(["event_id"])
            .withColumnRenamed("ingestion_timestamp", "bronze_ingestion_timestamp")
            .withColumn("silver_processing_timestamp", current_timestamp())
            .fillna(0.0, subset=["motor_current_amps", "temperature_c", "vibration_x"])
            .withColumn("est_power_watts", col("motor_current_amps") * lit(480.0))
            .withColumn("is_temp_outlier", col("temperature_c") > lit(85.0))
            .withColumn("is_vibration_outlier", col("vibration_x") > lit(2.5))
            .withColumn("machine_health_status", 
                when(col("is_temp_outlier") & col("is_vibration_outlier"), lit("CRITICAL"))
                .when(col("is_temp_outlier") | col("is_vibration_outlier"), lit("WARNING"))
                .otherwise(lit("HEALTHY"))
            )
        )
        
        # 6. IDEMPOTENT MERGE TO SILVER
        target_delta_table = DeltaTable.forName(spark, target_table_name)
        (target_delta_table.alias("target")
            .merge(
                df_enriched.alias("source"),
                "target.event_id = source.event_id"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
# MAGIC %md
# MAGIC ### Step 4: Execute the Silver Pipeline
# MAGIC Read incrementally from Bronze and apply the transformations via `foreachBatch`.

def run_silver_stream(stream_name: str):
    bronze_table = f"`{catalog_name}`.`{bronze_schema}`.{stream_name}_telemetry"
    silver_table = create_silver_table(stream_name)
    checkpoint_path = f"{checkpoint_base}{stream_name}/"
    
    print(f"Processing Silver stream for: {stream_name}")
    
    # Read incrementally from the Bronze Delta table
    return (spark.readStream
        .format("delta")
        .table(bronze_table)
        .writeStream
        .foreachBatch(lambda df, batchId: process_silver_microbatch(df, batchId, silver_table))
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True) # Runs as a micro-batch workflow
        .start()
    )

print("Starting Silver Transformation Pipeline...")

try:
    # Execute streams
    cnc_silver = run_silver_stream("cnc")
    conveyor_silver = run_silver_stream("conveyor")
    temp_silver = run_silver_stream("temperature")
    
    # Wait for completion
    cnc_silver.awaitTermination()
    conveyor_silver.awaitTermination()
    temp_silver.awaitTermination()
    
    # Best Practice: Optimize the tables for fast downstream querying
    print("Optimizing Silver Tables...")
    spark.sql(f"OPTIMIZE `{catalog_name}`.`{silver_schema}`.cnc_telemetry_clean ZORDER BY (device_id, event_time)")
    spark.sql(f"OPTIMIZE `{catalog_name}`.`{silver_schema}`.conveyor_telemetry_clean ZORDER BY (device_id, event_time)")
    spark.sql(f"OPTIMIZE `{catalog_name}`.`{silver_schema}`.temperature_telemetry_clean ZORDER BY (device_id, event_time)")
    
    print("✅ Silver pipeline completed successfully.")

except Exception as e:
    print(f"❌ CRITICAL PIPELINE FAILURE: {str(e)}")
    raise e